In [ ]:
# Check if they're selecting different subhalos/halos
import h5py

for ivol in [0, 500]:
    file_path = f'/cosma5/data/durham/dc-hick2/Galform_Out/L800/gp14/iz207/ivol{ivol}/galaxies.hdf5'
    with h5py.File(file_path, 'r') as f:
        output = f['Output001']
        # Check SubhaloID or ParticleID ranges
        if 'SubhaloID' in output:
            subhalo_ids = output['SubhaloID'][:]
            print(f"ivol{ivol} SubhaloIDs: min={subhalo_ids.min()}, max={subhalo_ids.max()}")
        if 'ParticleID' in output:
            particle_ids = output['ParticleID'][:]
            print(f"ivol{ivol} ParticleIDs: min={particle_ids.min()}, max={particle_ids.max()}")

In [ ]:
import h5py
import plotly.graph_objects as go

# Load the galaxies files
basepath = '/cosma5/data/durham/dc-hick2/Galform_Out/L800/gp14/'
redshift = 'iz207'
file_path_ivol0 = f'{basepath}{redshift}/ivol0/galaxies.hdf5'
file_path_ivol500 = f'{basepath}{redshift}/ivol500/galaxies.hdf5'

# Load ivol0 data
with h5py.File(file_path_ivol0, 'r') as f0:
    group0 = f0['Output001']
    xgal0 = group0['xgal'][:]
    ygal0 = group0['ygal'][:]
    zgal0 = group0['zgal'][:]

# Load ivol500 data
with h5py.File(file_path_ivol500, 'r') as f500:
    group500 = f500['Output001']
    xgal500 = group500['xgal'][:]
    ygal500 = group500['ygal'][:]
    zgal500 = group500['zgal'][:]

# Create interactive 3D scatter plot
fig = go.Figure()

# Add ivol0 galaxies in blue
fig.add_trace(go.Scatter3d(
    x=xgal0, y=ygal0, z=zgal0,
    mode='markers',
    name='ivol0',
    marker=dict(size=2, color='blue', opacity=0.5)
))

# Add ivol500 galaxies in red
fig.add_trace(go.Scatter3d(
    x=xgal500, y=ygal500, z=zgal500,
    mode='markers',
    name='ivol500',
    marker=dict(size=2, color='red', opacity=0.5)
))

# Update layout with labels and title
fig.update_layout(
    title='Galaxy Positions: ivol0 (blue) & ivol500 (red)',
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z'
    ),
    width=1000,
    height=800
)

fig.show()

In [ ]:
from pathlib import Path
import numpy as np
import plotly.graph_objects as go
from analysis.correlation.correlation import _load_positions_from_hdf5
from config import get_base_dir

base_dir_path = Path(get_base_dir())

def load_positions(
    redshift,
    ivol_max=1024,
    box_limit=5.0,
    ivol_offset=0,
    centrals_only=False,
    mhalo_min=None,
):
    x_list = []
    y_list = []
    z_list = []
    missing = 0
    iz_path = str(base_dir_path / redshift)
    for ivol in range(ivol_max + 1):
        actual_ivol = ivol + ivol_offset
        try:
            pos, _ = _load_positions_from_hdf5(
                iz_path,
                actual_ivol,
                centrals_only=centrals_only,
                mhalo_min=mhalo_min,
            )
        except (FileNotFoundError, KeyError):
            missing += 1
            continue
        if pos.size == 0:
            continue
        if box_limit is not None:
            mask = (
                (pos[:, 0] >= 0)
                & (pos[:, 0] <= box_limit)
                & (pos[:, 1] >= 0)
                & (pos[:, 1] <= box_limit)
                & (pos[:, 2] >= 0)
                & (pos[:, 2] <= box_limit)
            )
            if np.any(mask):
                x_list.append(pos[mask, 0])
                y_list.append(pos[mask, 1])
                z_list.append(pos[mask, 2])
        else:
            x_list.append(pos[:, 0])
            y_list.append(pos[:, 1])
            z_list.append(pos[:, 2])
    if not x_list:
        return np.array([]), np.array([]), np.array([]), missing
    return (
        np.concatenate(x_list),
        np.concatenate(y_list),
        np.concatenate(z_list),
        missing,
    )

# Build arrays from all subvolumes for iz271 and iz207 in a 5 Mpc box
x271, y271, z271, missing_271 = load_positions("iz271", ivol_max=1024, box_limit=5.0)
x207, y207, z207, missing_207 = load_positions("iz207", ivol_max=1024, box_limit=5.0)

print(f"iz271: N={len(x271)} (missing files: {missing_271})")
print(f"iz207: N={len(x207)} (missing files: {missing_207})")

# Create interactive 3D scatter plot
fig = go.Figure()

# Add iz271 galaxies in blue
fig.add_trace(go.Scatter3d(
    x=x271, y=y271, z=z271,
    mode="markers",
    name="iz271",
    marker=dict(size=2, color="blue", opacity=0.5)
))

# Add iz207 galaxies in red
fig.add_trace(go.Scatter3d(
    x=x207, y=y207, z=z207,
    mode="markers",
    name="iz207",
    marker=dict(size=2, color="red", opacity=0.5)
))

# Update layout with labels and title
fig.update_layout(
    title="Galaxy Positions: iz271 (blue) & iz207 (red), 0-5 Mpc box",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z",
    ),
    width=1000,
    height=800,
)

fig.show()

In [ ]:
import plotly.graph_objects as go

# Create 3D density field
# Define grid parameters
n_bins = 100  # 100^3 grid for performance (can increase for more detail)
box_size = 552  # L800 simulation box size in Mpc/h

# Combine both ivol datasets
x_all = np.concatenate([xgal0, xgal500])
y_all = np.concatenate([ygal0, ygal500])
z_all = np.concatenate([zgal0, zgal500])

# Create 3D histogram (density field)
hist, edges = np.histogramdd(
    (x_all, y_all, z_all), 
    bins=n_bins, 
    range=[[0, box_size], [0, box_size], [0, box_size]]
)

# Log scale for better visualization
hist_log = np.log10(hist + 1)

# Create coordinate grids for the volume
X, Y, Z = np.meshgrid(edges[0][:-1], edges[1][:-1], edges[2][:-1], indexing='ij')

# Flatten arrays for scatter plot (only plot cells with galaxies)
threshold = 0.3  # Adjust to show more/less structure
mask = hist_log > threshold
x_plot = X[mask]
y_plot = Y[mask]
z_plot = Z[mask]
density_plot = hist_log[mask]

# Create 3D scatter plot with density-based coloring
fig = go.Figure(data=[go.Scatter3d(
    x=x_plot,
    y=y_plot,
    z=z_plot,
    mode='markers',
    marker=dict(
        size=3,
        color=density_plot,
        colorscale='viridis',  # Similar to cosmic web visualizations
        colorbar=dict(title="log₁₀(N + 1)"),
        opacity=0.6,
        line=dict(width=0)
    ),
    hovertemplate='X: %{x:.1f}<br>Y: %{y:.1f}<br>Z: %{z:.1f}<br>Density: %{marker.color:.2f}<extra></extra>'
)])

fig.update_layout(
    title=f'3D Cosmic Web Structure ({redshift})',
    scene=dict(
        xaxis_title='X [Mpc/h]',
        yaxis_title='Y [Mpc/h]',
        zaxis_title='Z [Mpc/h]',
        bgcolor='black',  # Black background like the reference image
        xaxis=dict(backgroundcolor="black", gridcolor="gray", showbackground=True),
        yaxis=dict(backgroundcolor="black", gridcolor="gray", showbackground=True),
        zaxis=dict(backgroundcolor="black", gridcolor="gray", showbackground=True),
    ),
    width=1000,
    height=1000,
    paper_bgcolor='black',
    font=dict(color='white')
)

fig.show()